## RBTC-192 -- addition of missing loans from static table initializationexposureinfo

In [0]:
%sql
INSERT INTO `p-cdh`.reference.initializationexposureinfo
SELECT * FROM `d-cdh`.reference.initializationexposureinfo_new_loans;

##DATA2-283 - Portfolio update in payment_qca due to lack of field SPET_ID

In [0]:
env="p"

In [0]:
dates=["2025-03-24","2025-03-21","2025-03-20","2025-03-19","2025-03-18","2025-03-17"]

In [0]:
payment_qca_0314 = spark.sql(f"""
                             select * from `p-cdh`.primaryactive.payment 
                             where sourcesystem="qca" and businessdate="2025-03-14"
                             """)
payment_qca_0314.createOrReplaceTempView("payment_qca_0314")

In [0]:
for date in dates:
    spark.sql(f"""
        MERGE INTO `{env}-cdh`.primaryactive.payment AS target
        USING (
            SELECT DISTINCT PaymentKey,PortfolioName
            FROM payment_qca_0314 
        ) AS source
        ON target.PaymentKey = source.PaymentKey
        AND target.sourcesystem = 'qca'
        AND target.businessdate = '{date}'
        
        WHEN MATCHED THEN 
            UPDATE SET target.PortfolioName = source.PortfolioName
    """)

In [0]:

for date in dates:
    new_payments = spark.sql(f"""
        SELECT DISTINCT paymentkey 
        FROM `{env}-cdh`.primaryactive.payment 
        WHERE sourcesystem = 'qca' AND businessdate = '{date}'
        
        EXCEPT 
        
        SELECT DISTINCT paymentkey 
        FROM payment_qca_0314 
    """)
    new_payments.createOrReplaceTempView("new_payments")

    fs_payment_leaf = spark.sql(f"""
        SELECT DISTINCT PAY_CODE,CASE_ID
        FROM `{env}-cdh`.landed.qca_fs_payment_leaf
        WHERE BusinessDate='{date}'
    """)
    fs_payment_leaf.createOrReplaceTempView("fs_payment_leaf")

    fs_cases_snap = spark.sql(f"""
        SELECT DISTINCT CASE_CODE,SPET_CODE
        FROM `{env}-cdh`.landed.qca_fs_cases_snap
        WHERE BusinessDate='{date}'
    """)
    fs_cases_snap.createOrReplaceTempView("fs_cases_snap")

    portfolio = spark.sql(f"""
        SELECT DISTINCT PortfolioName,portfolio_id_qca
        FROM `{env}-cdh`.primaryactive.portfolio
        WHERE BusinessDate='{date}'                  
    """)
    portfolio.createOrReplaceTempView("portfolio")

    dim_cases_static = spark.sql(f"""
        SELECT DISTINCT  CASE_CODE
        FROM `{env}-cdh`.landed.qca_dim_cases_static
        WHERE BusinessDate='{date}'                  
    """)
    dim_cases_static.createOrReplaceTempView("dim_cases_static")

    update_df = spark.sql(f"""              
        SELECT DISTINCT l.PAY_CODE,p.PortfolioName FROM fs_payment_leaf l 
        INNER JOIN dim_cases_static cs 
        ON l.CASE_ID = cs.CASE_CODE
        INNER JOIN fs_cases_snap snp 
        on snp.CASE_CODE = cs.CASE_CODE
        INNER JOIN new_payments np 
        on np.PaymentKey = l.PAY_CODE
        LEFT JOIN portfolio p 
        on p.portfolio_id_qca = snp.SPET_CODE
    """)
    update_df.createOrReplaceTempView("update_df")

    update_query = f"""
        MERGE INTO `{env}-cdh`.primaryactive.payment target 
        USING update_df as source
        ON target.PaymentKey = source.PAY_CODE
        AND target.BusinessDate  = '{date}'
        AND target.sourcesystem = 'qca'
        WHEN MATCHED THEN UPDATE
        SET target.PortfolioName = source.PortfolioName
    """
    spark.sql(update_query)

##DD-179

### Steps
* Run the following code
* Re-run contact_details_arotron,contact_details_qc,contact_details_qca for all businessdates

In [0]:

spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.contactdetails
ADD COLUMN street1 STRING AFTER Street;
""")

In [0]:

spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.contactdetails
ADD COLUMN street2 STRING AFTER street1;
""")

In [0]:

spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.contactdetails
ADD COLUMN street3 STRING AFTER street2;
""")

In [0]:

spark.sql(f"""
MERGE INTO `d-cdh`.primaryactive.contactdetails target
USING `d-cdh`.primaryactive.contactdetails source
ON source.ContactDetailId = target.ContactDetailId
AND target.sourcesystem="profits" AND source.sourcesystem="profits"
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET
target.Street1 = source.Street
""")

## DATA2-75
### Steps
* Run the following code

In [0]:
from pyspark.sql.functions import col, when, lit, rtrim, row_number, sha2, concat_ws, coalesce
from pyspark.sql.window import Window

In [0]:
customer = spark.sql(f""" SELECT 
                        CUST_ID,
                        unit_code,
                        businessdate,
                        CDI,
                        CUST_TYPE,
                        CUST_STATUS,
                        STATUS_DATE,
                        FIRST_NAME,
                        LATIN_FIRSTNAME,
                        SURNAME,
                        LATIN_SURNAME,
                        FATHER_NAME,
                        MOTHER_NAME,
                        NAME_STANDARD,
                        TITLE,
                        SEX,
                        DATE_OF_BIRTH,
                        TAX_NUMBER,
                        ID_NO,
                        ISSUE_DATE,
                        EXPIRE_DATE,
                        gd_CRRAT,
                        gd_BHFMR,
                        businessdate      
                FROM `{env}-cdh`.landed.profits_customer
    """)

In [0]:
cust_address = spark.sql(f""" SELECT 
                            FK_CUSTOMERCUST_ID,
                            FKGD_HAS_COUNTRY,
                            BusinessDate
                FROM `{env}-cdh`.landed.profits_cust_address
    """)

In [0]:
other_id = spark.sql(f""" SELECT 
                        FK_CUSTOMERCUST_ID,
                        FKGD_HAS_TYPE,
                        ID_NO,
                        BusinessDate
                FROM `{env}-cdh`.landed.profits_other_id
    """)

In [0]:
generic_detail = spark.sql(f""" SELECT 
                            SERIAL_NUM,
                            PARAMETER_TYPE,
                            DESCRIPTION,
                            BusinessDate
                FROM `{env}-cdh`.landed.profits_generic_detail
    """)

In [0]:
customer_relation = spark.sql(f""" SELECT 
                            FST_CUST_ID,
                            sec_cust_id,
                            businessdate
                FROM `{env}-cdh`.landed.profits_customer_relation
    """)

In [0]:
organizationorigincountry_code = (
            customer.join(
                other_id,
                (other_id["FK_CUSTOMERCUST_ID"] == customer["CUST_ID"])
                & (other_id["FKGD_HAS_TYPE"] == 13)
                   & (other_id["BusinessDate"] == customer["businessdate"]),
                "left",
            )
            .join(
                cust_address, (cust_address["FK_CUSTOMERCUST_ID"] == customer["CUST_ID"])
                & (cust_address["BusinessDate"] == customer["businessdate"]), 
                "left",
            )
            .join(
                generic_detail, 
                (cust_address["FKGD_HAS_COUNTRY"] == generic_detail["SERIAL_NUM"])
                & (generic_detail["PARAMETER_TYPE"] == "CNTRY")
                & (generic_detail["BusinessDate"] == customer["businessdate"]), 
                "left",
            )
            .join(
                customer_relation,
                (customer["CUST_ID"] == customer_relation["sec_cust_id"])
                & (customer["BusinessDate"] == customer_relation["businessdate"]),
                "left",
            )
            .withColumn(
                "is_default", (col("DESCRIPTION") == "DEFAULT").cast("int")
            )
            .withColumn(
                "rank",
                row_number().over(
                    Window.partitionBy(customer["CUST_ID"]).orderBy(col("is_default"), generic_detail["DESCRIPTION"])
                    )
            )
            .withColumn(
                "PartyID",
                sha2(
                    concat_ws(";", 
                        customer["CUST_ID"], 
                        coalesce(customer_relation["FST_CUST_ID"], customer["CUST_ID"]), 
                        lit("profits")
                    ),
                    256
                )
            )
            .where((rtrim(other_id["ID_NO"]) != '')
                   & (customer["CUST_TYPE"] == 2)
                   )
            .select(
                customer["CUST_ID"].alias("CUST_ID_organizationcountrycode"),
                when(generic_detail["DESCRIPTION"] == "DEFAULT", lit(None)
                     ).otherwise(generic_detail["DESCRIPTION"]).alias("OrganizationOriginCountry"),
                customer["businessdate"].alias("BusinessDate"),
                col("PartyID"),
                ).distinct()
        )

organizationorigincountry_code = organizationorigincountry_code.filter("rank = 1").drop("rank")

organizationorigincountry_code.createOrReplaceTempView("organizationorigincountry_code")

In [0]:

spark.sql(f"""     
merge into `{env}-cdh`.primaryactive.party as target 
using organizationorigincountry_code as source
on source.PartyID = target.PartyID
    and target.sourcesystem="profits"
    and source.BusinessDate= target.BusinessDate

when matched then 
    update set 
        target.OrganizationOriginCountry = source.OrganizationOriginCountry
""")

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

